# Expected Shortfall (ES)

VaR has a major mathematical flaw: it is not a Coherent Risk Measure because it doesn't account for "Tail Risk": it only tells you where the door is, not how big the fire is behind it)

Expected Shortfall is simply the Conditional Mean.

The Heuristic: "If today is a 'Top 5% Bad Day,' what is the average loss I should expect?"

The Math: $ES_{\alpha} = E[X | X \le VaR_{\alpha}]$

In [4]:
import sys, os
import numpy as np
import pandas as pd
sys.path.append(os.path.abspath(os.path.join('..')))  # Add the parent directory to the path
from core.db_manager import RiskDBManager

# 1. Pull the data (Using our 'ID 2' logic from yesterday)
db = RiskDBManager()
query = "SELECT trade_date, adj_close FROM daily_metrics WHERE security_id = 2 ORDER BY trade_date ASC"
df = db.get_data(query)

# 2. Prep the returns
df['adj_close'] = df['adj_close'].astype(float)
df['returns'] = np.log(df['adj_close'] / df['adj_close'].shift(1))
returns = df['returns'].dropna()

# 3. Calculate 95% Historical VaR (The 'Gatekeeper')
var_95 = np.percentile(returns, 5)

# 4. Calculate Expected Shortfall (The 'Average Nightmare')
# We filter for only the returns that are WORSE than VaR
tail_events = returns[returns <= var_95]
es_95 = tail_events.mean()

print(f"95% Historical VaR: {var_95:.4%}")
print(f"95% Expected Shortfall: {es_95:.4%}")
print(f"95% Difference: {es_95 - var_95:.4%}")


95% Historical VaR: -2.7611%
95% Expected Shortfall: -4.1267%
95% Difference: -1.3656%


In [5]:
# Calculate 99% Historical VaR (The 1st Percentile / 5th worst day)
var_99 = np.percentile(returns, 1)

# Calculate 99% Expected Shortfall
es_99 = returns[returns <= var_99].mean()

print(f"99% Historical VaR: {var_99:.4%}")
print(f"99% Expected Shortfall: {es_99:.4%}")
print(f"99% Difference: {es_99 - var_99:.4%}")

99% Historical VaR: -4.9369%
99% Expected Shortfall: -6.4953%
99% Difference: -1.5584%


**LEPTOKURTOSIS** the ES/Var gap is widening from -1.37 to -1.56